# This notebook is for demonstrating some example use cases of the package

### The examples below also assume that you've already generated some runs using the techniques described in the README.md 

In [ ]:
import sys  
import numpy as np
import matplotlib.pyplot as plt
from src import BoidSimulator ,BoidVisualizer ,SimSaverLoader ,SimParams ,KernelReadout, FlatReadout

# Flat Readout Example

## No chunking or memory mapping

In [ ]:
# path to some collection of .npz files
run_paths = 'PATH/TO/NPZS'

#using a method from simsaverloader which loads then npzs dictionary into memory
datas = SimSaverLoader.find_npzs(run_paths)
print(datas[0].keys())

#as you can see from the print out, all the run data has been loaded successfully

In [ ]:
# Below i initialise a flatreadout object

# here I'm passing two of the dictionaries retrieved in the previous cell, corresponding to replica1 and replica2
# I'm also defining the washout, the period with which the lorenz is being sucked into its regular path
fr = FlatReadout(datas[0],datas[1],washout=1000,chunk_size=1000)

In [ ]:
#I can use that object to make predictions using the reservoir data
#but first I need to make some sort of readout

#this readout below is the most basic and just concatinates all the positions into one vector
state_vector = fr.get_reservoir_state_vectorised(fr.replica1)
state_vector2 = fr.get_reservoir_state_vectorised(fr.replica2)


#make a ridge prediction for t+0.5 (25*0.02) using the first replica split into testing and training
prediction, corr_coef = fr.ridge_prediction(state_vector,prediction_distance=25)

#alternitivly, use the first replica as the training set and the second as the testing set

#prediction, corr_coef = fr.ridge_prediction(state_vector,state_vector2)

#getting the returns so i can plot it

fr.plot_ridge_prediction(prediction,corr_coef,[2000,4000])

## Using Chunking and Memory Mapping
For very large runs or cost efficient readouts, I likely wont have enough memory to perform the calculations. Included in the readout classes is the option to use memory mapping and chunking

In [ ]:
#some of this stuff is explained in the previous section
run_paths = 'PATH/TO/NPZS'

# to do readouts and calculations using memory mapping, you have to load the data using memory mapping
datas = SimSaverLoader.find_npzs(run_paths,memory_map=True)

# everything else should be handled automatically
fr = FlatReadout(datas[0],datas[1],washout=1000,chunk_size=1000)

In [ ]:
state_vector = fr.get_reservoir_state_vectorised(fr.replica1)
prediction, corr_coef = fr.ridge_prediction(state_vector)
fr.plot_ridge_prediction(prediction,corr_coef,[2000,4000])

# Kernel Readout Example

In [ ]:
#some of this stuff is explained in the previous section
run_paths = 'PATH/TO/NPZS'
#using memory mapping (explained in previous section) because the kernel readouts are usually a lot more memory intensive
datas = SimSaverLoader.find_npzs(run_paths,memory_map=True)

# for the kernel readout, you need to specify the number of observation kernels you wish to generate
kr = KernelReadout(datas[0],datas[1],kernel_number=200,washout=1000,chunk_size=2000)

In [ ]:
#remember that the replica is just the dictionary of data representing a run, to perform calculations, we have to convert the data into some kind of readout vector
state_vector_1 = kr.get_reservoir_state_vectorised(kr.replica1)
state_vector_2 = kr.get_reservoir_state_vectorised(kr.replica2)

In [ ]:
# we can see how well this kernel readout predicts the lorenz
print(state_vector_2.shape)
prediction,corr_coef = kr.ridge_prediction(state_vector_2)
kr.plot_ridge_prediction(prediction,corr_coef,[0,2000])

In [ ]:
# we can also plot the consistent capacity

#note that there are a few different methods for calculating the consistent capacity
#the one below is most faithful to the methods outlined in the appendix of lymburn et al (2021)
cc, g2=kr.calc_consistency_profile(state_vector_1,state_vector_2,method='faithful')
kr.plot_consistency_profile(cc,g2,truncated_to=100)